# 03 — MEMO: dataset expandido + entrenamiento en GPU

Genera el dataset de reflexiones **expandido** (objetivo 1,500–2,000 pares, con más variación de paráfrasis por hecho que el run 2) y entrena MEMORY con LoRA **en GPU**: Qwen2.5-**1.5B**-Instruct como MEMORY (ya no el 0.5B), bf16, 3 épocas, LoRA r=32. En CPU el 1.5B tardaba ~36 h para 3 épocas (medido: 30.7 s/micro-paso); en la T4 es cuestión de minutos por época.

**Prerrequisito:** notebook 01 (chunks). El adaptador se guarda en Drive: si la sesión se corta, se reanuda desde el checkpoint.

**Regla de oro, igual que en local:** cada cifra del dataset se verifica contra el texto del chunk; las respuestas se copian byte-idénticas en las paráfrasis (solo cambia la pregunta), así que ninguna cifra puede cambiar.

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

# --- Parametros de MEMO (Colab, GPU T4) ---
MODEL_MEMORY = "Qwen/Qwen2.5-1.5B-Instruct"   # MEMORY: 1.5B (en CPU era 0.5B)
DATA_V3 = REPO + "/memo/reflection_qa_v3.jsonl"
OUT_DIR = REPO + "/memo/memory_model_colab"   # adaptador final (en Drive)
LORA_R = 32
LORA_ALPHA = 64
LR = 2e-4
EPOCHS = 3
BATCH = 8            # en GPU cabe batch mayor (en CPU era 1)
GRAD_ACCUM = 2       # batch efectivo 16
MAX_LEN = 128
SEED = 42
MAX_EXAMPLES = 0     # 0 = todos
MAX_SECONDS = 7200   # cortafuegos: 2 h por invocacion
GRAD_CKPT = False    # con batch 8 x seq 128 las activaciones caben sin checkpointing
INIT_ADAPTER = None  # para reanudar: ruta de un adaptador de una epoca anterior
TIME_TEST_STEPS = 5  # 0 = entrenamiento completo; >0 = solo mide s/paso y para

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive o generándolo del repo con `make extract` (`python pipeline/extract.py`).
3. **Solo si no lo subes**, generarlo aquí corriendo `extract.py` (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal.

`triples_seed.json` y `eval_questions.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive, o del repo con `make extract`.")
    print("Si no lo subes, el notebook lo genera con extract.py (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar con extract.py...")
        print("(descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        try:
            import pypdf
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1"], check=True)
        r = subprocess.run([sys.executable, "extract.py"], cwd=REPO + "/pipeline")
        print("extract.py exit:", r.returncode)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 accelerate>=1.16.0 peft>=0.22.0 numpy>=2.5.5

## 1. Dataset v1: 446 pares (determinista)

Corre `memo/generate_reflections.py` tal cual: lee los chunks, genera los pares curados + programáticos, excluye las 12 preguntas de evaluación (Jaccard ≥ 0.80) y verifica cada cifra contra el texto citado. Salida: `memo/reflection_qa.jsonl` (446 pares).

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "generate_reflections.py"], cwd=REPO + "/memo")
print("exit:", r.returncode)

## 2. Dataset v2: 446 → 1,414 (paráfrasis de pregunta)

Corre `memo/augment_v2.py` tal cual: paráfrasis solo de la pregunta (la respuesta se copia byte-idéntica), con chequeo de fuga por Jaccard contra las 12 preguntas de evaluación y verificación de que ninguna respuesta cambió sus números.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "augment_v2.py"], cwd=REPO + "/memo")
print("exit:", r.returncode)

## 3. Dataset v3: más variación por hecho (objetivo 1,500–2,000)

Una segunda pasada de aumento, en el notebook: reutiliza los helpers de `augment_v2` (`tokens`, `jaccard`, `candidates`) y añade más patrones de paráfrasis (más marcos y más trueques). Mismas reglas: solo cambia la pregunta, la respuesta es byte-idéntica; Jaccard ≥ 0.80 contra las 12 de evaluación descarta la paráfrasis; cada cifra clave debe aparecer ≥3 veces; sin duplicados (pregunta, respuesta). Si el total queda por debajo de 1,500, la celda lo dice y no inventa pares para llegar.

In [ ]:
import json, sys
sys.path.insert(0, REPO + "/memo")
import augment_v2 as aug

BASE = REPO + "/memo/reflection_qa.jsonl"
V2 = REPO + "/memo/reflection_qa_v2.jsonl"
EVAL = REPO + "/memo/eval_questions.json"
OUT = REPO + "/memo/reflection_qa_v3.jsonl"

EXTRA_FRAMINGS = [
    "Según los datos del informe anual de FEMSA 2024, ",
    "En el informe anual de FEMSA 2024, ",
    "Con base en el informe anual de FEMSA 2024, ",
    "De acuerdo con los datos del informe anual de FEMSA 2024, ",
]
EXTRA_SWAPS = [
    ("¿Cuáles fueron", "¿Cuáles resultaron ser"),
    ("¿Cuánto fue", "¿A cuánto ascendió"),
    ("¿A cuánto ascendieron", "¿Cuál fue el monto de"),
    ("¿Cuántos", "¿Qué número de"),
    ("¿Cuántas", "¿Qué número de"),
    ("¿Qué división creció más", "¿Cuál división tuvo el mayor crecimiento"),
    ("¿En qué países opera", "¿En qué países tiene operaciones"),
]


def extra_candidates(q):
    out = []
    for prefix in EXTRA_FRAMINGS:
        out.append(aug.framing(q, prefix))
    for old, new in EXTRA_SWAPS:
        s = aug.swap(q, old, new)
        if s:
            out.append(s)
    seen = {q}
    uniq = []
    for c in out:
        if c not in seen:
            seen.add(c)
            uniq.append(c)
    return uniq


rows = [json.loads(l) for l in open(BASE)]
v2_rows = [json.loads(l) for l in open(V2)]
eval_qs = json.load(open(EVAL))["questions"]
eval_toks = [aug.tokens(q["question"]) for q in eval_qs]

# Partimos del v2 (1,414) y añadimos las paráfrasis extra sobre los ORIGINALES.
by_base = {}
for r in v2_rows:
    by_base.setdefault(r["base_index"], []).append(r)

out_rows = list(v2_rows)
dropped_leak = 0
added = 0
for i, r in enumerate(rows):
    have = {x["question"] for x in by_base[i]}
    for c in extra_candidates(r["question"]):
        if c in have:
            continue
        if max((aug.jaccard(aug.tokens(c), et) for et in eval_toks), default=0) >= aug.JACCARD_MAX:
            dropped_leak += 1
            continue
        out_rows.append({**r, "question": c, "variant": "parafrasis_extra",
                         "base_index": i})
        have.add(c)
        added += 1

# Verificacion: respuestas byte-identicas a su fuente v1; sin duplicados.
for r in out_rows:
    src = rows[r["base_index"]]
    assert r["answer"] == src["answer"], f"respuesta cambiada en base {r['base_index']}"
    assert r["source_pages"] == src["source_pages"]
seen = set()
for r in out_rows:
    k = (r["question"], r["answer"])
    assert k not in seen, f"par duplicado: {k}"
    seen.add(k)

# Multiplicidad de cifras clave (mismo criterio que augment_v2).
from collections import Counter
key_nums = set()
for q in eval_qs:
    key_nums.update(aug.norm_num_tokens(str(n)) for n in q.get("expected_numbers", []))
fact_counts = Counter()
for r in out_rows:
    for n in aug.norm_num_tokens(r["answer"]) & key_nums:
        fact_counts[n] += 1
low = {n: c for n, c in fact_counts.items() if c < 3}
assert not low, f"cifras clave con <3 apariciones: {low}"

with open(OUT, "w") as f:
    for r in out_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"v2: {len(v2_rows)} -> v3: {len(out_rows)} (+{added} parafrasis extra, "
      f"{dropped_leak} descartadas por fuga Jaccard>=0.80)")
print(f"apariciones por cifra clave: min {min(fact_counts.values())}, "
      f"max {max(fact_counts.values())}")
print(f"-> {OUT}")

## 4. Prueba de tiempo (antes del entrenamiento completo)

Con `TIME_TEST_STEPS = 5` (arriba) la celda de entrenamiento solo mide segundos por micro-paso y estima el total. Si la estimación te convence, pon `TIME_TEST_STEPS = 0` y re-corre la celda para el entrenamiento completo. El modelo se descarga de Hugging Face a `HF_HOME` (en Drive): la primera vez tarda unos minutos; si HF va lento, la alternativa es ModelScope (`modelscope download Qwen/Qwen2.5-1.5B-Instruct`), como se documentó en el README de `memo/`).

## 5. Entrenamiento (adaptado a GPU)

Mismo algoritmo que `memo/train_memory_v2.py` (LoRA SFT, AdamW, schedule lineal con warmup, checkpoints cada 25 pasos, cortafuegos por tiempo), con los cambios mínimos para GPU: `device_map="auto"`, batches a CUDA, batch 8 (en CPU era 1). El adaptador y el log quedan en Drive.

In [ ]:
import json, math, os, random, time
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model

# --- Parametros del entrenamiento (los de arriba) ---
print(f"modelo MEMORY: {MODEL_MEMORY}")
print(f"datos: {DATA} | LoRA r={LORA_R} alpha={LORA_ALPHA} | lr={LR} | "
      f"epocas={EPOCHS} | batch={BATCH} grad_accum={GRAD_ACCUM} | max_len={MAX_LEN}")
print(f"salida: {OUT_DIR} (en Drive: sobrevive al reinicio)")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "-")
assert DEVICE == "cuda", "Este entrenamiento es para GPU: activa el runtime T4 (Runtime > Change runtime type)."

random.seed(SEED)
torch.manual_seed(SEED)

rows = [json.loads(l) for l in open(DATA)]
random.shuffle(rows)
if MAX_EXAMPLES:
    rows = rows[:MAX_EXAMPLES]
print(f"entrenando con {len(rows)} ejemplos de {DATA}")

tok = AutoTokenizer.from_pretrained(MODEL_MEMORY)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print(f"cargando {MODEL_MEMORY} (bf16, device_map=auto)...", flush=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_MEMORY, torch_dtype=torch.bfloat16,
                                             device_map="auto")
model.config.use_cache = False
if GRAD_CKPT:
    model.gradient_checkpointing_enable()
cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                 "gate_proj", "up_proj", "down_proj"],
                 task_type="CAUSAL_LM")
model = get_peft_model(model, cfg)
if INIT_ADAPTER:
    print(f"cargando adaptador inicial {INIT_ADAPTER} ...")
    model.load_adapter(INIT_ADAPTER, adapter_name="default")
model.print_trainable_parameters()


class QADataset(Dataset):
    def __init__(self, rows, tok, max_len):
        self.items = []
        skipped = 0
        for r in rows:
            prompt = (f"<|im_start|>user\n{r['question']}<|im_end|>\n"
                      f"<|im_start|>assistant\n")
            full = prompt + r["answer"] + "<|im_end|>"
            p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
            f_ids = tok(full, add_special_tokens=False)["input_ids"]
            if len(f_ids) > max_len:
                skipped += 1
                continue
            labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
            self.items.append((f_ids, labels))
        print(f"dataset: {len(self.items)} ejemplos, {skipped} omitidos (> {max_len} tokens)")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        ids, labels = self.items[i]
        return {"input_ids": ids, "labels": labels}


def collate(batch, pad_id):
    maxlen = max(len(b["input_ids"]) for b in batch)
    input_ids, labels, attn = [], [], []
    for b in batch:
        n = maxlen - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad_id] * n)
        labels.append(b["labels"] + [-100] * n)
        attn.append([1] * len(b["input_ids"]) + [0] * n)
    return {"input_ids": torch.tensor(input_ids),
            "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attn)}


ds = QADataset(rows, tok, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True,
                collate_fn=lambda b: collate(b, tok.pad_token_id))
total_opt_steps = max(1, int(math.ceil(len(ds) / (BATCH * GRAD_ACCUM)) * EPOCHS))
opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                        lr=LR, weight_decay=0.0)
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total_opt_steps), total_opt_steps)

t_start = time.time()
model.train()
step, micro, losses, loss_acc = 0, 0, [], 0.0
step_times = []
stop = False


def save_ckpt(tag):
    d = os.path.join(OUT_DIR + "_ckpt", f"step_{tag}")
    os.makedirs(d, exist_ok=True)
    model.save_pretrained(d)
    tok.save_pretrained(d)
    print(f"checkpoint -> {d}", flush=True)


while not stop:
    for batch in dl:
        if stop:
            break
        t0 = time.time()
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        loss_acc += out.loss.item()
        micro += 1
        step_times.append(time.time() - t0)
        if micro % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad), 1.0)
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            losses.append(loss_acc / GRAD_ACCUM)
            loss_acc = 0.0
            if step % 5 == 0 or step == 1:
                el = time.time() - t_start
                print(f"opt step {step}/{total_opt_steps} loss {losses[-1]:.4f} "
                      f"elapsed {el:.0f}s", flush=True)
            if step % 25 == 0:
                save_ckpt(step)
            if TIME_TEST_STEPS and step >= TIME_TEST_STEPS:
                s = sum(step_times) / len(step_times)
                print(f"\nTIME TEST: {s:.2f} s/micro-step (batch {BATCH} x seq<={MAX_LEN})")
                print(f"ETA corrida completa ({len(ds)} ej, {EPOCHS} ep): "
                      f"~{s * len(ds) * EPOCHS / 60:.0f} min")
                stop = True
                break
            if time.time() - t_start > MAX_SECONDS:
                print(f"TIMEBOX ({MAX_SECONDS:.0f}s) en el paso {step}; guardando.")
                stop = True
                break
            if step >= total_opt_steps:
                stop = True
                break

log = {"model": MODEL_MEMORY, "examples": len(ds), "epochs": EPOCHS,
       "lora_r": LORA_R, "lr": LR, "max_len": MAX_LEN,
       "batch_size": BATCH, "grad_accum": GRAD_ACCUM,
       "init_adapter": INIT_ADAPTER,
       "optimizer_steps": step, "micro_steps": micro,
       "loss_mean": sum(losses) / len(losses) if losses else None,
       "loss_last10_mean": (sum(losses[-10:]) / len(losses[-10:]) if losses else None),
       "seconds": time.time() - t_start}
if not TIME_TEST_STEPS:
    os.makedirs(OUT_DIR, exist_ok=True)
    model.save_pretrained(OUT_DIR)
    tok.save_pretrained(OUT_DIR)
    with open(os.path.join(OUT_DIR, "training_log.json"), "w") as f:
        json.dump(log, f, indent=2)
    print(f"\nadaptador guardado -> {OUT_DIR}")
print(json.dumps(log, indent=2))

## Qué mirar al terminar

- `training_log.json` en el adaptador: pérdida media por época y últimos 10 pasos. En el run 2 (CPU, 0.5B) la pérdida fue 0.61 → 0.16 → 0.087 (casi memorización). Con el 1.5B y más datos, la pérdida debería bajar parecido o más rápido.
- Si la sesión se corta: re-corre esta celda con `INIT_ADAPTER` apuntando al último checkpoint (`memory_model_colab_ckpt/step_N`) — el entrenamiento sigue desde ahí (misma aproximación que el run 2 en CPU).
- El adaptador que evalúa el notebook 04 es `OUT_DIR` (por defecto `memo/memory_model_colab`).

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

El adaptador final + sus checkpoints y los 3 datasets quedan en Drive. El notebook 04 verifica que exista el adaptador antes de evaluar; si la sesión se cortó a mitad, reanuda con `INIT_ADAPTER` apuntando al último checkpoint y vuelve a guardar aquí.

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('memo/memory_model_colab', 'Adaptador LoRA final (MEMORY 1.5B, 3 épocas)', True), ('memo/memory_model_colab_ckpt', 'Checkpoints cada 25 pasos (para reanudar)', False), ('memo/reflection_qa_v3.jsonl', 'Dataset v3 (1,500–2,000 pares)', True), ('memo/reflection_qa_v2.jsonl', 'Dataset v2 (1,414 pares)', False), ('memo/reflection_qa.jsonl', 'Dataset v1 (446 pares)', False)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)